# ShivAI Model Fine-Tuning on Free Google Colab GPU

This notebook fine-tunes an open-weights instruction model (such as `unsloth/Llama-3.2-3B-Instruct` or `unsloth/Qwen2.5-7B-Instruct`) into **ShivAI** using **Unsloth & QLoRA**.

### Why this works:
- Runs on a **free T4 GPU** on Google Colab.
- 2x faster, 70% less VRAM with Unsloth.
- Embeds ShivAI's persona, system identity, and coding rigor directly into the model weights.
- Exports GGUF for local Ollama / vLLM hosting or uploads to Hugging Face.

In [ ]:
# 1. Install Unsloth & Dependencies
!pip install --upgrade --no-cache-dir pip
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers" "trl<0.9.0" peft accelerate bitsandbytes

In [ ]:
# 2. Load Base Model with 4-bit Quantization
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
model_name = "unsloth/Llama-3.2-3B-Instruct"  # Or "unsloth/Qwen2.5-7B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = None, # None for auto-detection
    load_in_4bit = True,
)

# 3. Add LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [ ]:
# 4. Format Dataset for ShivAI Instruction Tuning
from datasets import load_dataset

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request as ShivAI. Always maintain the ShivAI identity. Never claim to be ChatGPT or Llama.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

# Load ShivAI dataset
# Upload 'alpaca_tuning_dataset.json' from your local training/ folder
dataset = load_dataset("json", data_files="alpaca_tuning_dataset.json", split="train")

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input_text, output) + tokenizer.eos_token
        texts.append(text)
    return { "text" : texts }

dataset = dataset.map(formatting_prompts_func, batched = True)

In [ ]:
# 5. Train with SFTTrainer
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

In [ ]:
# 6. Test Inference
FastLanguageModel.for_inference(model)
inputs = tokenizer(
[
    alpaca_prompt.format(
        "Who are you and what is your purpose?",
        "",
        ""
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 128, use_cache = True)
print(tokenizer.batch_decode(outputs))

In [ ]:
# 7. Export GGUF for Ollama or vLLM Hosting
# model.save_pretrained_gguf("shivai_model", tokenizer, quantization_method = "q4_k_m")
# Push to Hugging Face:
# model.push_to_hub_gguf("your_username/shivai-3b-gguf", tokenizer, quantization_method = "q4_k_m", token = "hf_...")